# Woolworths NZ Meal Cost Optimizer

Find the cheapest Woolworths for your recipe by comparing ingredient prices across nearby stores.

## Setup
Ensure you have the required dependencies and that the Woolworths store data is available in the `data/` directory.

In [ ]:
import sys
import pandas as pd
import math
import requests
import time
import os
import asyncio
from pathlib import Path
from playwright.async_api import async_playwright

if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

%load_ext autoreload
%autoreload 2

# Ensure project root is in path to allow relative package imports
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from scripts.woolworths.woolworths_optimizer import geocode, haversine, load_and_filter_stores, get_ingredients, get_quantities, change_store, scrape_products, analyze_results

import nest_asyncio
nest_asyncio.apply()

# ═══════ YOUR INPUTS ═══════
USER_ADDRESS = "123 Queen Street, Auckland CBD, 1010"
DISH_NAME = "spaghetti bolognese"
MAX_DISTANCE_KM = 5
# ═══════════════════════════

## 1. Geocode and Filter Stores
Locate stores within the specified radius of your address.

In [ ]:
user_lat, user_lon = geocode(USER_ADDRESS)

# Debug and construct robust absolute path
notebook_dir = os.getcwd()
print(f"DEBUG: CWD is {notebook_dir}")
stores_csv_path = os.path.abspath(os.path.join(notebook_dir, '..', 'data', 'woolworths_stores.csv'))
print(f"DEBUG: Constructed path is {stores_csv_path}")

stores = load_and_filter_stores(user_lat, user_lon, stores_csv_path=stores_csv_path, max_dist_km=MAX_DISTANCE_KM).head(2)
ingredients = get_ingredients(DISH_NAME)
quantities = get_quantities(DISH_NAME)

print(f"Dish: {DISH_NAME.title()}")
print(f"Ingredients: {', '.join(ingredients)}")
print(f"Stores to check: {', '.join(stores['name'])}")

## 2. Run Scraper
This runs the scraper as a standalone process and streams the output in real-time.

In [ ]:
import subprocess

results_file = "../data/latest_results.csv"
# Ensure file doesn't exist so we know it's being regenerated
if os.path.exists(results_file):
    os.remove(results_file)

cmd = [sys.executable, "../scripts/woolworths/woolworths_optimizer.py", USER_ADDRESS, DISH_NAME, results_file]

# Run and stream output in real-time
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end='')
process.wait()

## 3. Analysis and Comparison
Load results and display tables.

In [ ]:
df = pd.read_csv(results_file)
summary, table = analyze_results(df, ingredients, DISH_NAME)

print("=" * 65)
print("COST COMPARISON (Cheapest items only)")
print("=" * 65)
print(summary.to_string())
print("\nPer-Ingredient Breakdown:")
display(table)